# BIO -> HuggingFace-ready Dataset Preparation (bert-base-cased)

Rebuilds the tokenized dataset that the training repo (`data_in_brief_server/deb_bert_base`)
actually trains on, starting from the current train/val/test JSONL splits.

Only `bert-base-cased` is prepared. The earlier version of this notebook also
prepared DeBERTa, RoBERTa and BioBERT; the training repo trains a single
configuration (`bert-base-cased` + BIO + baseline), so the other three
backbones produced ~300 MB of output nothing ever read. Adding one back is a
one-line change (see `CHECKPOINT` in Cell 0) but the output folder name has to
stay in sync with `data_dirname` in `configs/model/bert.yaml`.

## Why this notebook has to be re-run after every split change

The training code **never reads the `.jsonl` files**. It reads only the
tokenized Arrow dataset produced here, through `load_from_disk()` in
`src/training/common.py`. Editing `bio_dataset_split/*.jsonl` therefore changes
nothing on its own: the splits and the tokenized dataset drift apart silently,
and training keeps running on the old data until this notebook is re-run.

## Paths

Input — written by `bio_train_val_test_split.ipynb`:

    ../animalia_dataset/bio_dataset_split/{train,val,test}.jsonl

Output — read directly by the trainer, no manual copy step:

    ../../data_in_brief_server/deb_bert_base/bio_tagging/bio_dataset/
      label_mapping.json      <- label2id / id2label
      prep_summary.json       <- run report (documentation only, nothing reads it)
      bert-base-cased/        <- the DatasetDict the trainer loads
        dataset_dict.json
        train/  validation/  test/

That output path is not arbitrary. It is exactly what the training configs
compose at runtime:

| Config | Key | Value |
|---|---|---|
| `configs/scheme/bio.yaml` | `dataset_dir` | `bio_tagging/bio_dataset` |
| `configs/model/bert.yaml` | `data_dirname` | `bert-base-cased` |

The last cell re-reads both YAML files and asserts they still resolve to what
was just written — that check is what catches the drift described above.

Two details worth keeping in mind:

- The split file is named `val.jsonl` on disk, but the `DatasetDict` key must be
  `validation` — that is the key `common.py` indexes.
- Nothing is padded here. Sequences stay variable-length and
  `DataCollatorForTokenClassification` pads per batch at train time.

No model is loaded and no training happens in this notebook.

## Cell 0 – Imports and configuration

In [ ]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path

from datasets import Dataset, DatasetDict, load_from_disk
from transformers import AutoTokenizer

CHECKPOINT = "bert-base-cased"
MAX_LENGTH = 512

SPLIT_DIR = Path("../animalia_dataset/bio_dataset_split")
OUT_ROOT = Path("../../data_in_brief_server/bio_tagging/bio_dataset")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Folder name must match `data_dirname` in configs/model/bert.yaml. The
# replace() keeps hub ids with a namespace (e.g. "dmis-lab/x" -> "dmis-lab__x")
# from creating a nested directory.
CHECKPOINT_DIR = OUT_ROOT / CHECKPOINT.replace("/", "__")
LABEL_MAP_PATH = OUT_ROOT / "label_mapping.json"
SUMMARY_PATH = OUT_ROOT / "prep_summary.json"

missing = [f"{n}.jsonl" for n in ("train", "val", "test")
           if not (SPLIT_DIR / f"{n}.jsonl").exists()]
if missing:
    raise FileNotFoundError(f"missing split file(s) in {SPLIT_DIR.resolve()}: {missing}")

print("Checkpoint :", CHECKPOINT)
print("Splits from:", SPLIT_DIR.resolve())
print("Output root:", OUT_ROOT.resolve())
print("Dataset dir:", CHECKPOINT_DIR.resolve())

Checkpoint : bert-base-cased
Splits from: /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_annotation/animalia_dataset/bio_dataset_split
Output root: /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_in_brief_server/bio_tagging/bio_dataset
Dataset dir: /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_in_brief_server/bio_tagging/bio_dataset/bert-base-cased


## Cell 1 – Load train/val/test JSONL into a DatasetDict

Model-agnostic: the same `raw_datasets` feeds the label mapping (Cell 2) and the
tokenizer (Cell 3).

The per-record length assertion is cheap insurance. A record whose `tokens` and
`biotags` lists disagree would not crash during tokenization — `word_ids()`
would just index into the shorter list and silently mislabel the sentence, or
raise an `IndexError` thousands of examples later with no usable context.

In [2]:
def load_split(name):
    """Read one JSONL split into a Dataset, checking token/tag alignment."""
    path = SPLIT_DIR / f"{name}.jsonl"
    records = []
    with path.open(encoding="utf-8") as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if len(rec["tokens"]) != len(rec["biotags"]):
                raise ValueError(
                    f"{path.name} line {lineno}: {len(rec['tokens'])} tokens "
                    f"vs {len(rec['biotags'])} biotags"
                )
            records.append({
                "tokens": rec["tokens"],
                "biotags": rec["biotags"],
                "source": rec.get("source", ""),
            })
    return Dataset.from_list(records)


# "val.jsonl" on disk -> "validation" key, which is what common.py indexes.
raw_datasets = DatasetDict({
    "train": load_split("train"),
    "validation": load_split("val"),
    "test": load_split("test"),
})

total = sum(len(ds) for ds in raw_datasets.values())
for split, ds in raw_datasets.items():
    print(f"{split:10s} {len(ds):>9,} sentences  ({len(ds) / total:5.1%})")
print(f"{'total':10s} {total:>9,} sentences")

train        152,632 sentences  (79.8%)
validation    19,337 sentences  (10.1%)
test          19,368 sentences  (10.1%)
total        191,337 sentences


## Cell 2 – Build the label2id / id2label mapping

Labels do not depend on the tokenizer, so this runs once. `"O"` is pinned to id
0 and the rest sorted alphabetically, which makes the ids reproducible across
runs.

If a mapping already exists on disk it is compared before being overwritten.
This matters: the label set determines the size and the meaning of the model's
classification head, so a change here invalidates every checkpoint already in
`outputs/` — they would have to be retrained, not just re-evaluated.

In [3]:
all_tags = set()
for split in raw_datasets.values():
    for tags in split["biotags"]:
        all_tags.update(tags)

# "O" first, then everything else sorted -> stable, reproducible ids.
label_list = ["O"] + sorted(t for t in all_tags if t != "O")
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(f"Total labels: {len(label_list)}")
print(label_list)

if LABEL_MAP_PATH.exists():
    old = json.loads(LABEL_MAP_PATH.read_text(encoding="utf-8"))["label2id"]
    if old == label2id:
        print("\nLabel mapping unchanged - existing checkpoints stay compatible.")
    else:
        added = sorted(set(label2id) - set(old))
        removed = sorted(set(old) - set(label2id))
        moved = sorted(k for k in set(old) & set(label2id) if old[k] != label2id[k])
        print("\n*** LABEL MAPPING CHANGED ***")
        print("  added     :", added or "-")
        print("  removed   :", removed or "-")
        print("  re-indexed:", moved or "-")
        print("  -> any model trained on the old mapping must be retrained,")
        print("     and its saved metrics no longer mean anything.")

LABEL_MAP_PATH.write_text(
    json.dumps({"label2id": label2id, "id2label": id2label}, indent=2),
    encoding="utf-8",
)
print(f"\nSaved label mapping -> {LABEL_MAP_PATH.resolve()}")

Total labels: 41
['O', 'B-F_R_i', 'B-GSI', 'B-JOi', 'B-Lb', 'B-Li', 'B-Lp', 'B-N_R_i', 'B-Ri', 'B-T', 'B-Ww0', 'B-Wwb', 'B-Wwi', 'B-Wwp', 'B-Wwx', 'B-ab', 'B-am', 'B-tR', 'B-tg', 'B-tp', 'B-tx', 'I-F_R_i', 'I-GSI', 'I-JOi', 'I-Lb', 'I-Li', 'I-Lp', 'I-N_R_i', 'I-Ri', 'I-T', 'I-Ww0', 'I-Wwb', 'I-Wwi', 'I-Wwp', 'I-Wwx', 'I-ab', 'I-am', 'I-tR', 'I-tg', 'I-tp', 'I-tx']

Label mapping unchanged - existing checkpoints stay compatible.

Saved label mapping -> /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_in_brief_server/bio_tagging/bio_dataset/label_mapping.json


## Cell 3 – Tokenize, realign the BIO tags onto subwords, save

BERT splits words into subwords, so the word-level tags have to be projected
onto the subword sequence. The rule: the **first** subword of a word carries
that word's label; every continuation subword and every special token
(`[CLS]`, `[SEP]`) gets `-100`, the index PyTorch's cross-entropy ignores.

Concretely, `cyprinid` tagged `O` becomes
`c / ##y / ##p / ##rini / ##d` -> `O / -100 / -100 / -100 / -100`, so the word
is scored once rather than five times.

`remove_columns` drops `tokens` / `biotags` / `source`: the training collator
expects only `input_ids`, `token_type_ids`, `attention_mask`, `labels`, and
leaving string columns in place makes it fail on collation.

Safe to re-run — the output folder is deleted first, so a re-run never mixes
old and new shards.

In [4]:
# add_prefix_space=True is a no-op for BERT's WordPiece tokenizer (it matters
# for byte-level BPE like RoBERTa); kept so this matches how
# src/inference/predict.py loads the tokenizer at inference time.
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT, add_prefix_space=True)
if not tokenizer.is_fast:
    raise ValueError(f"{CHECKPOINT} has no fast tokenizer - word_ids() alignment needs one.")


def tokenize_and_align_labels(batch):
    tokenized = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    all_labels = []
    for i, tags in enumerate(batch["biotags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:               # [CLS] / [SEP]
                label_ids.append(-100)
            elif word_id != prev_word_id:     # first subword of a word
                label_ids.append(label2id[tags[word_id]])
            else:                             # continuation subword
                label_ids.append(-100)
            prev_word_id = word_id
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized


tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

# No padding on purpose: DataCollatorForTokenClassification pads per batch.
if CHECKPOINT_DIR.exists():
    shutil.rmtree(CHECKPOINT_DIR)
tokenized_datasets.save_to_disk(str(CHECKPOINT_DIR))

print(f"\nSaved -> {CHECKPOINT_DIR.resolve()}")
for split, ds in tokenized_datasets.items():
    print(f"  {split:10s} {len(ds):>9,} examples")

Map:   0%|          | 0/152632 [00:00<?, ? examples/s]

Map:   0%|          | 0/19337 [00:00<?, ? examples/s]

Map:   0%|          | 0/19368 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/152632 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/19337 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/19368 [00:00<?, ? examples/s]


Saved -> /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_in_brief_server/bio_tagging/bio_dataset/bert-base-cased
  train        152,632 examples
  validation    19,337 examples
  test          19,368 examples


## Cell 4 – Verify what landed on disk

Reloads through the same `load_from_disk()` call the trainer uses, then checks
the structural contract: split names, column names, row counts. The subword
table underneath is the alignment spot-check — read down the `label` column and
confirm the entity's `B-` tag sits on the first subword of the right word.

In [5]:
reloaded = load_from_disk(str(CHECKPOINT_DIR))

EXPECTED_SPLITS = {"train", "validation", "test"}
EXPECTED_COLUMNS = {"input_ids", "token_type_ids", "attention_mask", "labels"}

assert set(reloaded) == EXPECTED_SPLITS, f"splits: {set(reloaded)} != {EXPECTED_SPLITS}"
for split, ds in reloaded.items():
    assert set(ds.column_names) == EXPECTED_COLUMNS, f"{split} columns: {ds.column_names}"
    assert len(ds) == len(raw_datasets[split]), f"{split}: {len(ds)} != {len(raw_datasets[split])}"
print("Structure OK:", EXPECTED_SPLITS, "x", sorted(EXPECTED_COLUMNS))

lengths = [len(x) for x in reloaded["train"][:5000]["input_ids"]]
print(f"Subword length (first 5,000 train rows): min {min(lengths)}, max {max(lengths)}, "
      f"mean {sum(lengths) / len(lengths):.1f}  [cap {MAX_LENGTH}]")

# Alignment spot-check on the first training sentence that carries an entity.
idx = next(i for i, tags in enumerate(raw_datasets["train"]["biotags"])
           if any(t.startswith("B-") for t in tags))
raw, enc = raw_datasets["train"][idx], reloaded["train"][idx]

print(f"\nSentence #{idx}: {' '.join(raw['tokens'])}")
print(f"Entity tags: {[t for t in raw['biotags'] if t != 'O']}\n")
print(f"{'subword':>16s}  {'label':>10s}")
print("-" * 30)
for tok_id, lab in zip(enc["input_ids"], enc["labels"]):
    sub = tokenizer.convert_ids_to_tokens(tok_id)
    print(f"{sub:>16s}  {(id2label[lab] if lab != -100 else '-100'):>10s}")

Structure OK: {'validation', 'test', 'train'} x ['attention_mask', 'input_ids', 'labels', 'token_type_ids']
Subword length (first 5,000 train rows): min 13, max 98, mean 40.7  [cap 512]

Sentence #2: The ultimate maximum length , representing the theoretical upper bound of somatic growth , is considerably larger at 8 cm , a value that highlights the substantial growth potential this species retains even in the cold , food - limited waters of the sub - Antarctic .
Entity tags: ['B-Li', 'I-Li']

         subword       label
------------------------------
           [CLS]        -100
             The           O
        ultimate           O
         maximum           O
          length           O
               ,           O
    representing           O
             the           O
     theoretical           O
           upper           O
           bound           O
              of           O
              so           O
         ##matic        -100
          growth           O
      

## Cell 5 – Cross-check against the training configs, then write the run summary

This is the cell that would have caught the stale-dataset problem. It re-reads
`configs/scheme/bio.yaml` and `configs/model/bert.yaml`, composes the path the
trainer will actually open, and compares it to what this notebook just wrote.

`prep_summary.json` is documentation only — no code reads it — but it records
which splits produced which dataset, so a future stale copy is diagnosable
instead of a guess.

In [6]:
CONFIG_DIR = Path("../../data_in_brief_server/configs")
scheme_path = CONFIG_DIR / "scheme" / "bio.yaml"
model_path = CONFIG_DIR / "model" / "bert.yaml"

if scheme_path.exists() and model_path.exists():
    import yaml

    scheme_cfg = yaml.safe_load(scheme_path.read_text(encoding="utf-8"))
    model_cfg = yaml.safe_load(model_path.read_text(encoding="utf-8"))
    repo_root = CONFIG_DIR.parent

    checks = [
        ("dataset dir  ",
         (repo_root / scheme_cfg["dataset_dir"] / model_cfg["data_dirname"]).resolve(),
         CHECKPOINT_DIR.resolve()),
        ("label mapping",
         (repo_root / scheme_cfg["label_mapping"]).resolve(),
         LABEL_MAP_PATH.resolve()),
        ("checkpoint   ", model_cfg["checkpoint"], CHECKPOINT),
    ]
    for name, from_config, from_notebook in checks:
        ok = from_config == from_notebook
        print(f"[{'OK ' if ok else 'FAIL'}] {name} -> {from_config}")
        if not ok:
            print(f"        notebook wrote -> {from_notebook}")
else:
    print(f"Training configs not found under {CONFIG_DIR.resolve()} - cross-check skipped.")

summary = [{
    "checkpoint": CHECKPOINT,
    "is_fast_tokenizer": tokenizer.is_fast,
    "vocab_size": tokenizer.vocab_size,
    "max_length": MAX_LENGTH,
    "num_labels": len(label_list),
    "source_split_dir": str(SPLIT_DIR.resolve()),
    "output_dir": str(CHECKPOINT_DIR.resolve()),
    "prepared_at": datetime.now(timezone.utc).isoformat(),
    **{f"n_{split}": len(ds) for split, ds in reloaded.items()},
}]
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"\nSummary saved -> {SUMMARY_PATH.resolve()}")
print(json.dumps(summary[0], indent=2))

[OK ] dataset dir   -> /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_in_brief_server/bio_tagging/bio_dataset/bert-base-cased
[OK ] label mapping -> /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_in_brief_server/bio_tagging/bio_dataset/label_mapping.json
[OK ] checkpoint    -> bert-base-cased

Summary saved -> /Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_in_brief_server/bio_tagging/bio_dataset/prep_summary.json
{
  "checkpoint": "bert-base-cased",
  "is_fast_tokenizer": true,
  "vocab_size": 28996,
  "max_length": 512,
  "num_labels": 41,
  "source_split_dir": "/Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_annotation/animalia_dataset/bio_dataset_split",
  "output_dir": "/Users/somaya/Documents/Phd/github_version/version_3/animalia-Github/animalia-dataset_pipeline/data_in_bri

## Next step — retrain

The checkpoint in `outputs/bio/bert/baseline/` was fitted on the *previous*
splits, and the metrics saved next to it (including the F1 in the training
repo's README) describe the previous test set. Because the splits were
re-drawn, sentences the old model trained on can now sit in the new test set,
so scoring that checkpoint against this dataset would leak. It has to be
retrained from scratch, not re-evaluated:

```bash
sbatch slurm/train/bio/train_bert_baseline.sbatch
```

Then refresh the result table in the training repo's `README.md` from the new
`outputs/bio/bert/baseline/test_metrics.json`.